In [60]:
# First cell - Imports and setup
import os
import sys
import pandas as pd
import warnings
import logging
import asyncio
from typing import Dict, Any
from datetime import datetime
from pathlib import Path

# Setup logging
logging.basicConfig(level=logging.INFO,
                   format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Suppress warnings
warnings.filterwarnings("ignore")

# Create performance_imgs directory
Path("performance_imgs").mkdir(exist_ok=True)

In [61]:
# Second cell - Get databases and initialize containers
root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
dbs = [db_path for db_path in os.listdir(os.path.join(root_path, "data", "live_bot_databases")) 
       if db_path.endswith('.sqlite')]

all_metrics = []
all_trades = []

In [62]:
def calculate_metrics(df: pd.DataFrame, controller_config: Dict[str, Any], side: int = 1):
    side_key = "grid_config_base" if side == 1 else "grid_config_quote"
    total_amount_quote = controller_config["total_amount_quote"]
    global_pnl = df["global_pnl"].iloc[-1]
    max_draw_down = df["global_pnl"].min() / total_amount_quote
    max_run_up = df["global_pnl"].max() / total_amount_quote
    total_trades = len(df)
    total_quote_volume = df["quote_amount"].sum()
    total_duration_minutes = (df["timestamp"].max() - df["timestamp"].min()) / 60
    metrics = {
        "global_pnl": global_pnl,
        "max_draw_down": max_draw_down,
        "max_run_up": max_run_up,
        "total_trades": total_trades,
        "total_quote_volume": total_quote_volume,
        "total_duration_minutes": total_duration_minutes,
        "controller_config": controller_config
    }
    return metrics

In [67]:
# Third cell - Modified with better error handling

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(root_path)

from core.data_sources.hummingbot_database import HummingbotDatabase
import research_notebooks.statarb_v2.stat_arb_performance_utils as utils
import numpy as np

for db_name in dbs:
    try:
        logger.info(f"Processing database: {db_name}")
        
        # Connect to database and get data
        db = HummingbotDatabase(db_name=db_name, root_path=root_path)
        executors_df = db.get_executors_data()
        executors_df["db_name"] = db_name
        
        # Get controller data
        controllers = db.get_controller_data().to_dict(orient="records")
        valid_controllers = [c for c in controllers if c["config"]["controller_name"] == "stat_arb"]
        
        logger.info(f"Found {len(valid_controllers)} valid controllers in {db_name}")
        
        # Process each controller
        for controller in valid_controllers:
            controller_executors = executors_df[executors_df["controller_id"] == controller["id"]]
            logger.info(f"Processing controller {controller['id']} with {len(controller_executors)} executors")
            
            # Process trades for this controller
            controller_trades = []
            for _, executor in controller_executors.iterrows():
                try:
                    custom_info = executor["custom_info"]
                    executor_config = executor["config"]
                    
                    if "filled_orders" not in custom_info:
                        logger.warning(f"No filled orders found for executor {executor['id']}")
                        continue
                        
                    for order_filled in custom_info["filled_orders"]:
                        for _, fill in order_filled["order_fills"].items():
                            trade_type = order_filled["trade_type"]
                            position_action = order_filled["position"]
                            position_multiplier = 1 if (trade_type == "BUY" and position_action == "OPEN") or \
                                                    (trade_type == "SELL" and position_action == "CLOSE") else -1

                            fill_dict = {
                                "db_name": executor["db_name"],
                                "controller_id": executor["controller_id"],
                                "side": executor_config["side"],
                                "trading_pair": order_filled["trading_pair"],
                                "order_type": order_filled["order_type"],
                                "trade_type": trade_type,
                                "cumulative_fee_paid_quote": sum([float(flat_fee["amount"]) 
                                                                for flat_fee in fill["fee"]["flat_fees"]]),
                                "position_action": position_action,
                                "timestamp": utils.ensure_timestamp_in_seconds(fill["fill_timestamp"]),
                                "price": float(fill["fill_price"]),
                                "base_amount": float(fill["fill_base_amount"]),
                                "quote_amount": float(fill["fill_quote_amount"]),
                                "position_multiplier": position_multiplier
                            }
                            controller_trades.append(fill_dict)
                except Exception as e:
                    logger.error(f"Error processing executor {executor['id']}: {str(e)}")
                    continue

            # if controller_trades:
            #     logger.info(f"Found {len(controller_trades)} trades for controller {controller['id']}")
            #     controller_trades_df = pd.DataFrame(controller_trades)
                
            #     # Create directory for this controller's images
            #     controller_dir = Path("performance_imgs") / controller["id"]
            #     controller_dir.mkdir(parents=True, exist_ok=True)

            #     metrics = {
            #         "controller_id": controller["id"],
            #         "db_name": db_name
            #     }

            #     # Process long side
            #     long_df = utils.calculate_performance_metrics(controller_trades_df, controller["id"], side=1)
            #     if not long_df.empty:
            #         logger.info(f"Processing long side metrics for controller {controller['id']}")
            #         long_metrics = calculate_metrics(long_df, controller["config"], side=1)
            #         metrics.update({f"long_{k}": v for k, v in long_metrics.items()})
                    
            #         # Generate and save long performance chart
            #         config = controller_executors[controller_executors['config'].apply(lambda x: x['side']) == 1]['config'].iloc[0]
            #         long_fig = await utils.plot_candles_with_global_pnl_chart(
            #             long_df, 
            #             side=1,
            #             top_value=config['end_price'],
            #             bottom_value=config['start_price']
            #         )
            #         long_fig.write_image(str(controller_dir / "long_performance.jpg"))

            #     # Process short side
            #     short_df = utils.calculate_performance_metrics(controller_trades_df, controller["id"], side=2)
            #     if not short_df.empty:
            #         logger.info(f"Processing short side metrics for controller {controller['id']}")
            #         short_metrics = calculate_metrics(short_df, controller["config"], side=2)
            #         metrics.update({f"short_{k}": v for k, v in short_metrics.items()})
                    
            #         # Generate and save short performance chart
            #         config = controller_executors[controller_executors['config'].apply(lambda x: x['side']) == 2]['config'].iloc[0]
            #         short_fig = await utils.plot_candles_with_global_pnl_chart(
            #             short_df,
            #             side=2,
            #             top_value=config['end_price'],
            #             bottom_value=config['start_price']
            #         )
            #         short_fig.write_image(str(controller_dir / "short_performance.jpg"))

            #     if metrics:
            #         logger.info(f"Adding metrics for controller {controller['id']}")
            #         all_metrics.append(metrics)
            #         all_trades.extend(controller_trades)


            # if controller_trades:
            #     logger.info(f"Found {len(controller_trades)} trades for controller {controller['id']}")
            #     controller_trades_df = pd.DataFrame(controller_trades)
                
            #     # Get start and end datetime from trades
            #     start_timestamp = controller_trades_df['timestamp'].min()
            #     end_timestamp = controller_trades_df['timestamp'].max()
                
            #     # Convert timestamps to datetime
            #     start_datetime = datetime.fromtimestamp(start_timestamp)
            #     end_datetime = datetime.fromtimestamp(end_timestamp)
                
            #     # Create directory for this controller's images
            #     controller_dir = Path("performance_imgs") / controller["id"]
            #     controller_dir.mkdir(parents=True, exist_ok=True)

            #     # Extract trading pairs from controller config
            #     base_pair = controller["config"]["base_trading_pair"]
            #     quote_pair = controller["config"]["quote_trading_pair"]
            #     trading_pairs = f"{base_pair}+{quote_pair}"

            #     metrics = {
            #         "controller_id": controller["id"],
            #         "db_name": db_name,
            #         "trading_pairs": trading_pairs,
            #         "start_datetime": start_datetime,
            #         "end_datetime": end_datetime,
            #         "duration_hours": (end_datetime - start_datetime).total_seconds() / 3600  # Added duration in hours
            #     }

            #     # Process long side
            #     long_df = utils.calculate_performance_metrics(controller_trades_df, controller["id"], side=1)
            #     if not long_df.empty:
            #         logger.info(f"Processing long side metrics for controller {controller['id']}")
            #         long_metrics = calculate_metrics(long_df, controller["config"], side=1)
            #         metrics.update({f"long_{k}": v for k, v in long_metrics.items()})
                    
            #         # Calculate long side additional metrics
            #         long_config = controller["config"]["grid_config_base"]
            #         long_amount_of_levels = (controller["config"]["total_amount_quote"] / 2) / long_config["min_order_amount_quote"]
            #         long_spread_percent = abs(long_config["end_price"] - long_config["start_price"]) / long_amount_of_levels
                    
            #         metrics.update({
            #             "long_amount_of_levels": long_amount_of_levels,
            #             "long_spread_percent": long_spread_percent
            #         })
                    
            #         # Generate and save long performance chart
            #         config = controller_executors[controller_executors['config'].apply(lambda x: x['side']) == 1]['config'].iloc[0]
            #         long_fig = await utils.plot_candles_with_global_pnl_chart(
            #             long_df, 
            #             side=1,
            #             top_value=config['end_price'],
            #             bottom_value=config['start_price']
            #         )
            #         long_fig.write_image(str(controller_dir / "long_performance.jpg"))

            #     # Process short side
            #     short_df = utils.calculate_performance_metrics(controller_trades_df, controller["id"], side=2)
            #     if not short_df.empty:
            #         logger.info(f"Processing short side metrics for controller {controller['id']}")
            #         short_metrics = calculate_metrics(short_df, controller["config"], side=2)
            #         metrics.update({f"short_{k}": v for k, v in short_metrics.items()})
                    
            #         # Calculate short side additional metrics
            #         short_config = controller["config"]["grid_config_quote"]
            #         short_amount_of_levels = (controller["config"]["total_amount_quote"] / 2) / short_config["min_order_amount_quote"]
            #         short_spread_percent = abs(short_config["end_price"] - short_config["start_price"]) / short_amount_of_levels
                    
            #         metrics.update({
            #             "short_amount_of_levels": short_amount_of_levels,
            #             "short_spread_percent": short_spread_percent
            #         })
                    
            #         # Generate and save short performance chart
            #         config = controller_executors[controller_executors['config'].apply(lambda x: x['side']) == 2]['config'].iloc[0]
            #         short_fig = await utils.plot_candles_with_global_pnl_chart(
            #             short_df,
            #             side=2,
            #             top_value=config['end_price'],
            #             bottom_value=config['start_price']
            #         )
            #         short_fig.write_image(str(controller_dir / "short_performance.jpg"))

            #     if metrics:
            #         logger.info(f"Adding metrics for controller {controller['id']}")
            #         all_metrics.append(metrics)
            #         all_trades.extend(controller_trades)
            if controller_trades:
                logger.info(f"Found {len(controller_trades)} trades for controller {controller['id']}")
                controller_trades_df = pd.DataFrame(controller_trades)
                
                # Get start and end datetime from trades
                start_timestamp = controller_trades_df['timestamp'].min()
                end_timestamp = controller_trades_df['timestamp'].max()
                start_datetime = datetime.fromtimestamp(start_timestamp)
                end_datetime = datetime.fromtimestamp(end_timestamp)
                
                # Create directory for this controller's images
                controller_dir = Path("performance_imgs") / controller["id"]
                controller_dir.mkdir(parents=True, exist_ok=True)

                # Extract trading pairs from controller config
                base_pair = controller["config"]["base_trading_pair"]
                quote_pair = controller["config"]["quote_trading_pair"]
                trading_pairs = f"{base_pair}+{quote_pair}"

                metrics = {
                    "controller_id": controller["id"],
                    "db_name": db_name,
                    "trading_pairs": trading_pairs,
                    "start_datetime": start_datetime,
                    "end_datetime": end_datetime,
                    "duration_hours": (end_datetime - start_datetime).total_seconds() / 3600
                }

                # Process long side
                long_df = utils.calculate_performance_metrics(controller_trades_df, controller["id"], side=1)
                if not long_df.empty:
                    logger.info(f"Processing long side metrics for controller {controller['id']}")
                    long_metrics = calculate_metrics(long_df, controller["config"], side=1)
                    metrics.update({f"long_{k}": v for k, v in long_metrics.items()})
                    
                    # Calculate long side additional metrics
                    long_config = controller["config"]["grid_config_base"]
                    
                    # Calculate amount of levels based on order amount
                    long_amount_levels_by_amount = (controller["config"]["total_amount_quote"] / 2) / long_config["min_order_amount_quote"]
                    long_amount_levels_by_spread = np.nan
                    # Calculate amount of levels based on min spread if it exists
                    if "min_spread_between_orders" in controller["config"]:
                        long_amount_levels_by_spread = abs(long_config["end_price"] - long_config["start_price"]) / controller["config"]["min_spread_between_orders"]
                        # Use the minimum of both calculations
                        long_amount_of_levels = min(long_amount_levels_by_amount, long_amount_levels_by_spread)
                    else:
                        long_amount_of_levels = long_amount_levels_by_amount
                        
                    # Calculate spread percent using the final amount of levels
                    long_spread_percent = abs(long_config["end_price"] - long_config["start_price"]) / long_amount_of_levels
                    
                    metrics.update({
                        "long_amount_of_levels": long_amount_of_levels,
                        "long_spread_percent": long_spread_percent,
                        "long_amount_levels_by_spread" : long_amount_levels_by_spread,
                        "long_amount_levels_by_amount" : long_amount_levels_by_amount,
                    })
                    
                    # Generate and save long performance chart...

                # Process short side
                short_df = utils.calculate_performance_metrics(controller_trades_df, controller["id"], side=2)
                if not short_df.empty:
                    logger.info(f"Processing short side metrics for controller {controller['id']}")
                    short_metrics = calculate_metrics(short_df, controller["config"], side=2)
                    metrics.update({f"short_{k}": v for k, v in short_metrics.items()})
                    
                    # Calculate short side additional metrics
                    short_config = controller["config"]["grid_config_quote"]
                    
                    # Calculate amount of levels based on order amount
                    short_amount_levels_by_amount = (controller["config"]["total_amount_quote"] / 2) / short_config["min_order_amount_quote"]
                    
                    # Calculate amount of levels based on min spread if it exists
                    if "min_spread_between_orders" in controller["config"]:
                        short_amount_levels_by_spread = abs(short_config["end_price"] - short_config["start_price"]) / controller["config"]["min_spread_between_orders"]
                        # Use the minimum of both calculations
                        short_amount_of_levels = min(short_amount_levels_by_amount, short_amount_levels_by_spread)
                    else:
                        short_amount_of_levels = short_amount_levels_by_amount
                        
                    # Calculate spread percent using the final amount of levels
                    short_spread_percent = abs(short_config["end_price"] - short_config["start_price"]) / short_amount_of_levels
                    
                    metrics.update({
                        "short_amount_of_levels": short_amount_of_levels,
                        "short_spread_percent": short_spread_percent,
                        "short_amount_levels_by_spread" : short_amount_levels_by_spread,
                        "short_amount_levels_by_amount" : short_amount_levels_by_amount,
                    })
                    
                    # Generate and save short performance chart...

                if metrics:
                    logger.info(f"Adding metrics for controller {controller['id']}")
                    all_metrics.append(metrics)
                    all_trades.extend(controller_trades)

            else:
                logger.warning(f"No trades found for controller {controller['id']}")

    except Exception as e:
        logger.error(f"Error processing database {db_name}: {str(e)}")
        continue

# Create final DataFrames
metrics_df = pd.DataFrame(all_metrics)
trades_df = pd.DataFrame(all_trades)

# Print summary
print("\nFinal Summary:")
print(f"Total metrics collected: {len(all_metrics)}")
print(f"Total trades collected: {len(all_trades)}")
if not metrics_df.empty:
    print("\nMetrics DataFrame columns:", metrics_df.columns.tolist())
    print("\nSample metrics:")
    print(metrics_df.head())
else:
    print("\nNo metrics collected!")

# Save results
if not metrics_df.empty:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    metrics_df.to_csv(f"performance_summary_{timestamp}.csv", index=False)
    trades_df.to_csv(f"all_trades_{timestamp}.csv", index=False)
    print(f"\nResults saved as performance_summary_{timestamp}.csv and all_trades_{timestamp}.csv")

metrics_df

2025-03-14 17:59:23,430 - INFO - Processing database: binance_perpetual-2025-11-3-0010-batch3-2025.sqlite
2025-03-14 17:59:23,764 - INFO - Found 8 valid controllers in binance_perpetual-2025-11-3-0010-batch3-2025.sqlite
2025-03-14 17:59:23,768 - INFO - Processing controller binance-perpetual||ADA-USDT||RED-USDT||2025||isoweek11||step-0.0010_3-0010 with 4 executors
2025-03-14 17:59:23,777 - INFO - Found 1987 trades for controller binance-perpetual||ADA-USDT||RED-USDT||2025||isoweek11||step-0.0010_3-0010
2025-03-14 17:59:23,792 - INFO - Processing long side metrics for controller binance-perpetual||ADA-USDT||RED-USDT||2025||isoweek11||step-0.0010_3-0010
2025-03-14 17:59:23,796 - INFO - Processing short side metrics for controller binance-perpetual||ADA-USDT||RED-USDT||2025||isoweek11||step-0.0010_3-0010
2025-03-14 17:59:23,797 - INFO - Adding metrics for controller binance-perpetual||ADA-USDT||RED-USDT||2025||isoweek11||step-0.0010_3-0010
2025-03-14 17:59:23,798 - INFO - Processing contr


Final Summary:
Total metrics collected: 48
Total trades collected: 52752

Metrics DataFrame columns: ['controller_id', 'db_name', 'trading_pairs', 'start_datetime', 'end_datetime', 'duration_hours', 'long_global_pnl', 'long_max_draw_down', 'long_max_run_up', 'long_total_trades', 'long_total_quote_volume', 'long_total_duration_minutes', 'long_controller_config', 'long_amount_of_levels', 'long_spread_percent', 'long_amount_levels_by_spread', 'long_amount_levels_by_amount', 'short_global_pnl', 'short_max_draw_down', 'short_max_run_up', 'short_total_trades', 'short_total_quote_volume', 'short_total_duration_minutes', 'short_controller_config', 'short_amount_of_levels', 'short_spread_percent', 'short_amount_levels_by_spread', 'short_amount_levels_by_amount']

Sample metrics:
                                       controller_id  \
0  binance-perpetual||ADA-USDT||RED-USDT||2025||i...   
1  binance-perpetual||ADA-USDT||RED-USDT||2025||i...   
2  binance-perpetual||ADA-USDT||RED-USDT||2025||i.

,controller_id,db_name,trading_pairs,start_datetime,end_datetime,duration_hours,long_global_pnl,long_max_draw_down,long_max_run_up,long_total_trades,...,short_max_draw_down,short_max_run_up,short_total_trades,short_total_quote_volume,short_total_duration_minutes,short_controller_config,short_amount_of_levels,short_spread_percent,short_amount_levels_by_spread,short_amount_levels_by_amount
0,binance-perpetual||ADA-USDT||RED-USDT||2025||i...,binance_perpetual-2025-11-3-0010-batch3-2025.s...,ADA-USDT+RED-USDT,2025-03-12 00:18:50,2025-03-12 09:58:40,9.66388889,6.20945903,-0.00601507,0.00676259,143,...,-0.0847126,0.00376703,1844,12886.127,579.83333333,"{'controller_name': 'stat_arb', 'controller_ty...",76.92307692,0.00081975,157.64480056,76.92307692
1,binance-perpetual||ADA-USDT||RED-USDT||2025||i...,binance_perpetual-2025-11-3-0010-batch3-2025.s...,ADA-USDT+RED-USDT,2025-03-12 00:18:50,2025-03-12 09:58:37,9.66305556,6.30770418,-0.005955,0.00686084,152,...,-0.08438504,0.00399384,1865,12972.9383,579.78333333,"{'controller_name': 'stat_arb', 'controller_ty...",76.92307692,0.00081975,157.64480056,76.92307692
2,binance-perpetual||ADA-USDT||RED-USDT||2025||i...,binance_perpetual-2025-11-3-0010-batch3-2025.s...,ADA-USDT+RED-USDT,2025-03-12 00:18:50,2025-03-12 09:58:37,9.66305556,6.25177673,-0.00597094,0.00684751,152,...,-0.08397635,0.00392092,1847,12956.9121,579.78333333,"{'controller_name': 'stat_arb', 'controller_ty...",76.92307692,0.00081975,157.64480056,76.92307692
3,binance-perpetual||ADA-USDT||RED-USDT||2025||i...,binance_perpetual-2025-11-3-0010-batch3-2025.s...,ADA-USDT+RED-USDT,2025-03-12 00:18:50,2025-03-12 09:58:41,9.66416667,6.26109619,-0.00596242,0.00686593,145,...,-0.08452786,0.00397923,1860,12946.1182,579.85,"{'controller_name': 'stat_arb', 'controller_ty...",76.92307692,0.00081975,157.64480056,76.92307692
4,binance-perpetual||ADA-USDT||RED-USDT||2025||i...,binance_perpetual-2025-11-3-0010-batch3-2025.s...,ADA-USDT+RED-USDT,2025-03-12 00:18:50,2025-03-12 09:58:40,9.66388889,6.20945903,-0.00601507,0.00676259,143,...,-0.0847126,0.00376703,1844,12886.127,579.83333333,"{'controller_name': 'stat_arb', 'controller_ty...",76.92307692,0.00081975,157.64480056,76.92307692
5,binance-perpetual||ADA-USDT||RED-USDT||2025||i...,binance_perpetual-2025-11-3-0010-batch3-2025.s...,ADA-USDT+RED-USDT,2025-03-12 00:18:50,2025-03-12 09:58:37,9.66305556,6.30770418,-0.005955,0.00686084,152,...,-0.08438504,0.00399384,1865,12972.9383,579.78333333,"{'controller_name': 'stat_arb', 'controller_ty...",76.92307692,0.00081975,157.64480056,76.92307692
6,binance-perpetual||ADA-USDT||RED-USDT||2025||i...,binance_perpetual-2025-11-3-0010-batch3-2025.s...,ADA-USDT+RED-USDT,2025-03-12 00:18:50,2025-03-12 09:58:37,9.66305556,6.25177673,-0.00597094,0.00684751,152,...,-0.08397635,0.00392092,1847,12956.9121,579.78333333,"{'controller_name': 'stat_arb', 'controller_ty...",76.92307692,0.00081975,157.64480056,76.92307692
7,binance-perpetual||ADA-USDT||RED-USDT||2025||i...,binance_perpetual-2025-11-3-0010-batch3-2025.s...,ADA-USDT+RED-USDT,2025-03-12 00:18:50,2025-03-12 09:58:41,9.66416667,6.26109619,-0.00596242,0.00686593,145,...,-0.08452786,0.00397923,1860,12946.1182,579.85,"{'controller_name': 'stat_arb', 'controller_ty...",76.92307692,0.00081975,157.64480056,76.92307692
8,binance-perpetual||BOME-USDT||SWARMS-USDT||202...,binance_perpetual-2025-11-3-0010-batch1-2025.s...,BOME-USDT+SWARMS-USDT,2025-03-12 00:19:31,2025-03-12 07:10:50,6.85527778,11.24289989,-0.00746037,0.01180109,277,...,-0.02030615,0.01215493,759,5304.0639,411.31666667,"{'controller_name': 'stat_arb', 'controller_ty...",17.70692056,0.0004,17.70692056,76.92307692
9,binance-perpetual||BOME-USDT||SWARMS-USDT||202...,binance_perpetual-2025-11-3-0010-batch1-2025.s...,BOME-USDT+SWARMS-USDT,2025-03-12 00:19:31,2025-03-12 06:53:38,6.56861111,10.88175169,-0.00775831,0.01143899,266,...,-0.02023218,0.0114088,731,5221.972,394.11666667,"{'controller_name': 'stat_arb', 'controller_ty...",17.70692056,0.0004,17.706920

In [68]:
metrics_df[[col for col in metrics_df.columns if 'long' in col]]

,long_global_pnl,long_max_draw_down,long_max_run_up,long_total_trades,long_total_quote_volume,long_total_duration_minutes,long_controller_config,long_amount_of_levels,long_spread_percent,long_amount_levels_by_spread,long_amount_levels_by_amount
0,6.20945903,-0.00601507,0.00676259,143,1070.1284,294.43333333,"{'controller_name': 'stat_arb', 'controller_ty...",76.92307692,0.00211043,405.8522774,76.92307692
1,6.30770418,-0.005955,0.00686084,152,1097.7055,294.43333333,"{'controller_name': 'stat_arb', 'controller_ty...",76.92307692,0.00211043,405.8522774,76.92307692
2,6.25177673,-0.00597094,0.00684751,152,1099.1042,294.43333333,"{'controller_name': 'stat_arb', 'controller_ty...",76.92307692,0.00211043,405.8522774,76.92307692
3,6.26109619,-0.00596242,0.00686593,145,1100.5232,294.6,"{'controller_name': 'stat_arb', 'controller_ty...",76.92307692,0.00211043,405.8522774,76.92307692
4,6.20945903,-0.00601507,0.00676259,143,1070.1284,294.43333333,"{'controller_name': 'stat_arb', 'controller_ty...",76.92307692,0.00211043,405.8522774,76.92307692
5,6.30770418,-0.005955,0.00686084,152,1097.7055,294.43333333,"{'controller_name': 'stat_arb', 'controller_ty...",76.92307692,0.00211043,405.8522774,76.92307692
6,6.25177673,-0.00597094,0.00684751,152,1099.1042,294.43333333,"{'controller_name': 'stat_arb', 'controller_ty...",76.92307692,0.00211043,405.8522774,76.92307692
7,6.26109619,-0.00596242,0.00686593,145,1100.5232,294.6,"{'controller_name': 'stat_arb', 'controller_ty...",76.92307692,0.00211043,405.8522774,76.92307692
8,11.24289989,-0.00746037,0.01180109,277,2036.705548,331.25,"{'controller_name': 'stat_arb', 'controller_ty...",0.48946836,0.0004,0.48946836,76.92307692
9,10.88175169,-0.00775831,0.01143899,266,1995.28502,331.25,"{'controller_name': 'stat_arb', 'controller_ty...",0.48946836,0.0004,0.48946836,76.92307692


In [65]:
{ 'grid_config_base': {'start_price': 1.8837351876086172,
    'end_price': 2.281264812391383,
    'limit_price': 1.784352781412926,
    'min_order_amount_quote': 6.5,
    'order_frequency': 5},
}
{
'grid_config_base': {'start_price': 1.8837351876086172,
    'end_price': 2.281264812391383,
    'limit_price': 1.784352781412926,
    'min_order_amount_quote': 6.5,
    'order_frequency': 5},}

{'grid_config_base': {'start_price': 1.8837351876086172,
  'end_price': 2.281264812391383,
  'limit_price': 1.784352781412926,
  'min_order_amount_quote': 6.5,
  'order_frequency': 5}}

In [66]:
from core.data_sources.clob import CLOBDataSource
clob = CLOBDataSource()
from core.services.mongodb_client import MongoClient
from core.services.backend_api_client import BackendAPIClient

# mongo_client = MongoClient(
#     username=os.getenv("MONGO_INITDB_ROOT_USERNAME", "admin"),
#     password=os.getenv("MONGO_INITDB_ROOT_PASSWORD", "admin"),
#     host=os.getenv("MONGO_HOST", "localhost"),
#     port=os.getenv("MONGO_PORT", 27017),
#     database=os.getenv("MONGO_DATABASE", "quants_lab")
# )

# uri = f"mongodb://{os.getenv('MONGO_INITDB_ROOT_USERNAME', 'admin')}:{os.getenv('MONGO_INITDB_ROOT_PASSWORD', 'admin')}@{os.getenv('MONGO_HOST', 'localhost')}:{os.getenv('MONGO_PORT', '27017')}/{os.getenv('MONGO_DATABASE', 'quants_lab')}"
# mongo_client = MongoClient(uri)

connector_name = 'binance_perpetual'
CONNECTOR_INSTANCE = clob.get_connector(connector_name)

# CLOBDataSource 
# config['base_trading_pair'] = config["trading_pair"]
# 'grid_config_base'
prices = await utils.get_executor_prices(executor_config_dict = executor_config, connector_instance = CONNECTOR_INSTANCE, side = 'long')
prices

2025-03-14 17:42:51,163 - INFO - Initializing ClobDataSource
2025-03-14 17:42:51,212 - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x2bbee8730>


KeyError: 'base_trading_pair'

In [ ]:
CONNECTOR_INSTANCE.trading_pairs

[]